In [0]:
from pyspark.sql import functions as F
from pyspark.sql.types import *
from pyspark.sql.window import *
from delta.tables import *
from pyspark.testing import assertDataFrameEqual

### Scenario:
So far every exercise has ended with `.show()` and a visual comparison against an expected table. Real pipeline code doesn't get validated that way — it gets covered by **unit tests** that run automatically, catch regressions before deployment, and don't require a human to eyeball output every time. Today's exercise is about writing a transformation function properly, then writing tests **against that function**, not just running it once and looking at the result.

**Problem — do all three parts:**

**Part 1 — Write the function:**
- Write a Python function `calculate_final_price(df)` that takes a DataFrame with columns `product_id` (string), `price` (double), `discount_pct` (double), and returns a new DataFrame with an added `final_price` column: `price * (1 - discount_pct / 100)`.
- The function should be a proper reusable function — takes a DataFrame in, returns a DataFrame out. No hardcoded file paths or reads inside it.

**Part 2 — Build test inputs directly in code (no file upload this time):**
- Since this is about testing logic, build your test DataFrames directly with `spark.createDataFrame(...)` inside your test code — don't read from a file.

**Part 3 — Write at least 4 test cases, each as its own function, asserting the output is exactly correct:**
- **Happy path** — a normal discount (e.g., price=100, discount_pct=20 → final_price=80.0).
- **Zero discount** — discount_pct=0 → final_price should equal price exactly.
- **Full discount** — discount_pct=100 → final_price should be 0.0.
- **Multiple rows at once** — a DataFrame with 3+ rows, confirming every row's `final_price` is correct, not just the first one.
- For each test, compare the actual output DataFrame against an expected DataFrame using `pyspark.testing.utils.assertDataFrameEqual` (if available in your runtime) — if it's not available, manually `.collect()` both DataFrames and assert the row lists are equal.
- Each test function should `print("PASSED: <test name>")` on success, and let the `assert` failure naturally surface if something's wrong — don't swallow it with try/except.

**Expected Behavior**
All 4 test functions should run and print `PASSED` with no assertion errors. There's no single "expected output table" for this one — the deliverable is working test code that actually validates the function's correctness, including the edge cases (0% and 100% discount) that a lazy test suite would skip.

In [0]:
def calculate_final_price(df):
    return (
        df.withColumn(
            "final_price",
            F.col("price") * (1 - F.col("discount_pct")/100)
        )
    )

def test_happy_path():
    input_df = spark.createDataFrame(
        [("P101", 1500.0, 10.0)],
        ["product_id", "price", "discount_pct"]
    )
    expected_df = spark.createDataFrame(
        [("P101", 1500.0, 10.0, 1350.0)],
        ["product_id", "price", "discount_pct", "final_price"]
    )
    actual_df = calculate_final_price(input_df)

    assertDataFrameEqual(actual_df, expected_df)
    print("PASSED: test_happy_path")

def test_zero_discount_path():
    input_df = spark.createDataFrame(
        [("P102", 580.0, 0.0)],
        ["product_id", "price", "discount_pct"]
    )
    expected_df = spark.createDataFrame(
        [("P102", 580.0, 0.0, 580.0)],
        ["product_id", "price", "discount_pct", "final_price"]
    )
    actual_df = calculate_final_price(input_df)

    assertDataFrameEqual(actual_df, expected_df)
    print("PASSED: test_zero_discount_path")

def test_full_discount_path():
    input_df = spark.createDataFrame(
        [("P103", 720.12, 100.0)],
        ["product_id", "price", "discount_pct"]
    )
    expected_df = spark.createDataFrame(
        [("P103", 720.12, 100.0, 0.0)],
        ["product_id", "price", "discount_pct", "final_price"]
    )
    actual_df = calculate_final_price(input_df)

    assertDataFrameEqual(actual_df, expected_df)
    print("PASSED: test_full_discount_path")

def test_multi_rows_path():
    input_df = spark.createDataFrame(
        [
            ("P104", 220.0, 15.0),
            ("P105", 1000.00, 12.0),
            ("P103", 2500.0, 20.0),
            ("P103", 3000.0, 30.0)
        ],
        ["product_id", "price", "discount_pct"]
    )
    expected_df = spark.createDataFrame(
        [
            ("P104", 220.0, 15.0, 187.0),
            ("P105", 1000.00, 12.0, 880.00),
            ("P103", 2500.0, 20.0, 2000.00),
            ("P103", 3000.0, 30.0, 2100.00)
        ],
        ["product_id", "price", "discount_pct", "final_price"]
    )
    actual_df = calculate_final_price(input_df)

    assertDataFrameEqual(actual_df, expected_df)
    print("PASSED: test_multi_rows_path")

In [0]:
test_happy_path()
test_zero_discount_path()
test_full_discount_path()
test_multi_rows_path()